In [0]:
s3_bucket =  f's3://vendor-performance-1511/data/'
catalog_name = 'vendor_performance'
schema_name = 'bronze'
files_data = dbutils.fs.ls(s3_bucket)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
for file in files_data:
    if '.csv' in file.name:
        # 1. Read the CSV from S3
        bronze_df = spark.read \
            .option("header", "true")\
            .option("delimiter", ",")\
            .option("header", "true")\
            .option("inferSchema", "true")\
            .csv(f"{s3_bucket}{file.name}") \
            .withColumn("file_name", F.col("_metadata.file_path")) \
            .withColumn("ingest_timestamp", F.current_timestamp())
        
        # 3. Write as a Delta Table (Managed Table)
        # display(bronze_df.limit(5))
        bronze_df.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable(f"{catalog_name}.{schema_name}.{file.name[:-4]}")

In [0]:
%sql
select * from vendor_performance.bronze.purchases limit 10